# Python Regex

> 📘 **Python Mastery** · Module 05 — Intermediate Python · Lesson 5/7

Regular expressions (regex) are a tiny language for describing *shapes of text* -- find it, extract it, replace it, validate it. One skill, endless mileage: logs, forms, scraped pages, model output.

## 🎯 Learning Objectives

- Explain what regular expressions are used for
- Choose between `findall`, `search`, `match`, `fullmatch`, `sub` and `split`
- Read and write patterns using metacharacters (`\d`, `\w`, `.`, `+`, `[...]`, ...)
- Control greediness with lazy quantifiers and pull out parts with groups
- Apply flags like `re.IGNORECASE` and reuse patterns with `re.compile`
- Solve everyday jobs: extract emails, normalise phones, validate usernames, clean whitespace

## 1. What Is Regex?

A regex is a **pattern** -- a formula describing text ("three digits, then a dash, then four digits"). You hand the pattern plus some text to the `re` module, and it tells you where the shape occurs.

Four classic jobs:
1. **Find** -- does this text contain a date?
2. **Extract** -- pull every email out of a page.
3. **Replace** -- mask card numbers with stars.
4. **Validate** -- is this a legal username?

**Syntax:**
```python
import re
re.findall(pattern, text)     # simplest entry point
```

**Example:** find every number hiding in a sentence.

In [ ]:
import re

note = "Table 4 needs 3 chairs, and room 42 needs 21."

print(re.findall(r"\d+", note))            # every run of digits
print(re.findall(r"[A-Za-z]+", note))      # every word

## 2. Raw Strings First: `r"..."`

Regex leans heavily on backslashes (`\d` means digit). But in ordinary Python strings a backslash starts escapes (`\n` = newline), so you would have to type every backslash twice: `"\\d"`. Prefix the string with `r` and Python takes it literally -- one backslash, exactly as typed.

You will see both styles in the wild; always prefer raw strings for patterns.

**Syntax:**
```python
r"\d+"          # raw: what you see is what the regex engine gets
"\\d+"           # same string value, but you had to fight for it
```

**Example:**

In [ ]:
plain = "\\d+\\.\\d+"       # escaped backslashes -- hard to read
raw   = r"\d+\.\d+"          # raw string -- same value, easy to read

print(plain == raw, "|", raw)

# Without raw strings, innocent-looking text bites:
# "C:\temp\new" hides a TAB (\t) and a NEWLINE (\n)!
print(r"C:\temp\new", "<- raw keeps backslashes honest")

## 3. The Workhorse Functions

Six functions cover almost everything. The subtlest difference: **`search` looks anywhere**, while **`match` only tries position 0**.

| Function | Where it looks | Returns |
|---|---|---|
| `re.findall(p, s)` | everywhere | list of matched strings |
| `re.search(p, s)` | first match, anywhere | Match object or `None` |
| `re.match(p, s)` | start of string only | Match object or `None` |
| `re.fullmatch(p, s)` | entire string must fit | Match object or `None` |
| `re.sub(p, repl, s)` | everywhere | new string with replacements |
| `re.split(p, s)` | everywhere | list cut at each match |

Match objects carry detail: `.group()` (the text), `.start()`, `.end()`, `.span()`.

> 🔍 **Under the Hood:** Python's `re` engine is a *backtracking* matcher: it commits to one path through the pattern and, when the text disagrees, rewinds and tries another branch. That design makes patterns fast to run but vulnerable to pathological cases -- nested quantifiers like `(a+)+` can force millions of rewind attempts on a long string. It also explains why lazy quantifiers sometimes win: less text consumed per attempt means fewer rewinds.

**Syntax:**
```python
m = re.search(pattern, text)
if m:                       # ALWAYS check before using .group()
    m.group()
```

**Example:**

In [ ]:
import re

log = "ERROR 404 at 12:01 | ok | ERROR 500 at 13:15"

print(re.findall(r"ERROR \d+", log))

m = re.search(r"\d{2}:\d{2}", log)             # first time stamp, anywhere
print("found", m.group(), "at index", m.start())

print("match sees it?", re.match(r"\d{2}:\d{2}", log) is not None)   # False!
print(re.sub(r"ERROR", "WARN", log))           # replace all
print(re.split(r" \| ", log))                  # cut on ' | '

## 4. Metacharacter Reference

The alphabet of patterns. Learn these twelve and you can read most real-world regexes:

| Symbol | Meaning | Matches in `"Room 42!"` |
|---|---|---|
| `\d` | a digit 0-9 | `4`, `2` |
| `\D` | NOT a digit | `R`, `o`, `o`, `m`, ` `, `!` |
| `\w` | letter, digit or underscore | `Room42` |
| `\W` | NOT a word character | ` `, `!` |
| `\s` | whitespace (space, tab, newline) | ` ` |
| `\S` | NOT whitespace | everything else |
| `.` | any single char except newline | each character |
| `^` | start of string/line | before `R` |
| `$` | end of string/line | after `!` |
| `*` | previous item, 0 or more times | |
| `+` | previous item, 1 or more times | `4` then `2` |
| `?` | previous item, 0 or 1 (optional) | |
| `{n,m}` | between n and m repetitions | `\d{2}` grabs `42` |
| `[abc]` | one char listed in the set | `a`, `b`, or `c` |
| `[^abc]` | one char NOT in the set | |
| `|` | either side (OR) | |
| `(...)` | group -- capture for later | |

Escape any special symbol with a backslash to mean the literal character: `\.` matches a real dot.

**Example:** quantifiers in action.

In [ ]:
import re

code = "ID-AB123 2026-08-26 x9 zz"

print(re.findall(r"[A-Z]{2}\d{3}", code))   # two capitals + three digits
print(re.findall(r"\d{4}-\d{2}-\d{2}", code))
print(re.findall(r"x\d*", code))            # star allows ZERO digits
print(re.findall(r"x\d+", code))            # plus demands at least one

## 5. Character Classes & Ranges

Square brackets define a menu of allowed characters: `[aeiou]`, ranges `[a-z]`, `[0-9]`, combinations `[0-9a-fA-F]`. A leading `^` flips it into "anything except": `[^0-9]`.

Inside brackets most symbols lose their magic -- no escaping needed.

**Syntax:**
```python
[aeiou]       # one lowercase vowel
[a-zA-Z0-9_]  # exactly what \w means
[^0-9]        # anything but a digit (same job as \D)
```

**Example:**

In [ ]:
import re

word = "beautiful day"
vowels = re.findall(r"[aeiou]", word)
print(vowels, "->", len(vowels), "vowels")

token = "fa3 G7 #zz"
print(re.findall(r"[0-9a-f]", token))          # hex-ish characters only

messy = "Room 42, floor #7!"
print(re.findall(r"[^a-zA-Z ]", messy))        # everything that isn't a letter/space

## 6. Quantifiers: Greedy vs Lazy

By default `*`, `+` and `{n,m}` are **greedy**: they grab as much text as possible, even swallowing your closing marker. Add `?` to make them **lazy** -- stop at the first opportunity instead.

This one extra character is the difference between parsing HTML-ish text successfully and mangling it.

**Syntax:**
```python
r"<td>(.*)</td>"     # greedy: runs to the LAST </td>
r"<td>(.*?)</td>"    # lazy:   stops at the FIRST </td>
```

**Example:**

In [ ]:
import re

row = "<td>Sarah</td><td>Dhaka</td>"

print(re.findall(r"<td>(.*)</td>", row))     # greedy grabs across both cells
print(re.findall(r"<td>(.*?)</td>", row))    # lazy keeps cells separate

## 7. Capturing Groups

Wrap part of a pattern in parentheses to **capture** it. After a successful match you can read each piece individually -- this is how regex *extracts structured fields* from free text.

- `m.group(0)` -- the whole match
- `m.group(1)`, `m.group(2)` -- first, second group
- `(?P<name>...)` -- named group, readable by keyword
- Back-references (`\1`) even work inside `sub` replacements

**Syntax:**
```python
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", text)
m.group(1)                                               # year
re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", text)    # shuffle the order
```

**Example:**

In [ ]:
import re

entry = "Parcel delivered on 2026-08-26 to Dhaka."

m = re.search(r"(\d{4})-(\d{2})-(\d{2})", entry)
print("whole :", m.group(0))
print("parts :", m.groups())            # ('2026', '08', '26')

named = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", entry)
print("month =", named.group("month"))

print(re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", entry))   # 26/08/2026

## 8. Flags: Changing the Rules

Optional behaviour arrives through flags, combinable with `|`:

| Flag | Effect |
|---|---|
| `re.IGNORECASE` | case no longer matters |
| `re.MULTILINE` | `^` and `$` apply at **every line break**, not just the text ends |
| `re.DOTALL` | `.` finally matches newlines too |

**Syntax:**
```python
re.findall(pattern, text, re.IGNORECASE)
re.findall(pattern, text, re.MULTILINE | re.DOTALL)   # combine with |
```

**Example:**

In [ ]:
import re

reply = "YES please.\nno thanks\nMAYBE later"
print(reply.replace("\n", " / "))
print(re.findall(r"\byes\b|\bno\b|\bmaybe\b", reply, re.IGNORECASE))

build_log = "INFO boot\nERROR disk full\nINFO login"
print("default ^   :", re.findall(r"^ERROR.*", build_log))
print("MULTILINE ^ :", re.findall(r"^ERROR.*", build_log, re.MULTILINE))

para = "<p>first line\nstill the same paragraph</p>"
print(re.search(r"<p>(.*?)</p>", para, re.DOTALL).group(1))

## 9. `re.compile`: Build Once, Use Everywhere

Compiling turns a pattern string into a pattern **object** with the same methods (`findall`, `search`, `sub`, ...). Do it once at module top-level for hot loops: clearer code (the pattern gets a NAME) and faster repeated matching, since the pattern is translated only once.

**Syntax:**
```python
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[A-Za-z]{2,}")
EMAIL.findall(text)
```

**Example:**

In [ ]:
import re

CAPITALISED = re.compile(r"\b[A-Z][a-z]+\b")     # proper nouns, roughly

quote = "Sarah met Arif near Dhaka University on a Tuesday."

print(CAPITALISED.findall(quote))

for hit in CAPITALISED.finditer(quote):          # positions included
    print(hit.group(), hit.span())

## 10. Practical Mini-Patterns

Four everyday jobs, each solved in a few lines. Steal these shapes.

### 10a. Extract email addresses

In [ ]:
import re

message = ("Write to sarah.rahman@gmail.com or arif+bhuyian@yahoo.co.uk today.\n"
           "Fragments like @nope.com or name@ are ignored.")

EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[A-Za-z]{2,}")
print(EMAIL.findall(message))

### 10b. Normalise phone numbers

In [ ]:
import re

contacts = ["+8801712345678", "017-1234-5678", "(02) 55667788", "not a phone"]

BD_MOBILE = re.compile(r"(?:88)?01[3-9]\d{8}")   # 11 digits, optional 88 prefix

for original in contacts:
    digits = re.sub(r"\D", "", original)          # strip EVERYTHING but digits
    verdict = "valid mobile" if BD_MOBILE.fullmatch(digits) else "rejected"
    print(f"{original:<16} -> {digits:<14} {verdict}")

### 10c. Validate a username

In [ ]:
import re

USERNAME = re.compile(r"[A-Za-z0-9_]{3,16}")     # 3-16 chars: letters, digits, _

candidates = ["sarah_k", "ab", "has space", "way_too_long_username_x", "Rafi2026"]

for name in candidates:
    ok = USERNAME.fullmatch(name) is not None
    print(f"{name:<24} {'accepted' if ok else 'REJECTED'}")

### 10d. Clean messy whitespace

In [ ]:
import re

review = "Best    biryani\tin town!!!\n\n     Highly   recommend."

single_spaced = re.sub(r"\s+", " ", review).strip()
print(single_spaced)

# bonus: remove the space BEFORE punctuation using a capture group
polished = re.sub(r"\s+([,.!?])", r"\1", single_spaced)
print(polished)

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Writing `"\d+"` without the `r` prefix | backslash fights string escapes; warnings and confusion | always raw strings: `r"\d+"` |
| Using `match()` hoping for search-anywhere | silently returns `None` unless the text starts with the pattern | `re.search` (anywhere) / `re.fullmatch` (validate) |
| Greedy `.*` | swallows up to the *last* occurrence of the closer | make it lazy: `.*?` |
| Unescaped `.` | matches ANY character, so pattern `3.14` also hits `3x14` | escape literal dots |
| One regex to validate ALL possible emails | the full spec is famously regex-impossible | pragmatic pattern for input hints; confirm with a verification mail |
| Nested quantifiers like `(a+)+` | catastrophic backtracking -- matching can hang forever | simplify the pattern; test against long inputs |

## 💡 Best Practices & Pro Tips

- Prefer plain string methods when they suffice: `startswith`, `in` and `replace` beat regex for simple checks -- readable wins.
- Name compiled patterns in UPPER_CASE at module top: `EMAIL_RE`, `DATE_RE`.
- Build and debug interactively on regex101.com (set the flavour to Python); paste real samples.
- For gnarly patterns use verbose mode `re.VERBOSE` and comment each piece inline.
- **AI-engineering relevance:** regex is the scalpel around LLMs -- extract JSON from code fences, scrub PII from prompts, mine dates and IDs from logs, and sanity-check model outputs cheaply before reaching for fancier tools.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `re.findall(p, s)` | all matches as a list | `findall(r"\d+", "a1 b22") -> ['1', '22']` |
| `re.search(p, s)` | first match anywhere | check `if m:` then `m.group()` |
| `re.match` / `re.fullmatch` | anchored at start / whole string | validation |
| `re.sub(p, repl, s)` | replace all | `sub(r"\s+", " ", s)` |
| `re.split(p, s)` | cut string at matches | `split(r",\s*", csv_line)` |
| `r"..."` | raw string -- always for patterns | `r"\d+\.\d+"` |
| `( )` / `(?P<n>...)` | capturing / named groups | `m.group("year")` |
| `re.I / re.M / re.S` | ignorecase, multiline, dotall | combinable with `|` |
| `re.compile(p)` | pre-built pattern object | `EMAIL.findall(text)` |

Key takeaways:

- Regex describes *shape*: classes say WHAT may appear, quantifiers say HOW OFTEN, anchors say WHERE.
- `search` finds, `match` anchors, `fullmatch` validates -- pick deliberately.
- Groups turn matching into extraction; laziness (`?`) tames greedy operators.

## 🔗 Next Lesson

Up next: **[06_PIP](../06_PIP/notes.ipynb)** -- installing the community's code: pip, PyPI and requirements files.